# 05. Исследовательский notebook: Kaggle Crypto News dataset

Цель ноутбука — сначала не запускать весь ML pipeline, а понять, насколько датасет `oliviervha/crypto-news` подходит для нашей задачи.

Проверяем:

1. какие файлы есть в датасете;
2. какие таблицы реально читаются;
3. есть ли date/timestamp;
4. есть ли title/text/body;
5. можно ли выделить BTC/ETH/SOL/BNB/XRP и другие монеты;
6. сколько наблюдений получается после агрегации по датам;
7. есть ли пересечение с price data из `yfinance`;
8. на каких активах и горизонтах потенциально лучше запускать RUN 05 pipeline.

После этого можно будет выбрать лучший asset/horizon и уже запускать FinBERT / embeddings / ablation.

## 0. Установка пакетов

В Colab эту ячейку лучше запустить. Локально можно пропустить, если пакеты уже установлены.

In [ ]:
%pip install -q kagglehub yfinance pandas numpy matplotlib pyarrow openpyxl

## 1. Импорты и настройки

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import kagglehub
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)

KAGGLE_DATASET = 'oliviervha/crypto-news'
OUTPUT_DIR = Path('artifacts/run_05_dataset_research')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ASSET_KEYWORDS = {
    'BTC': ['btc', 'bitcoin', 'xbt'],
    'ETH': ['eth', 'ethereum', 'ether'],
    'SOL': ['sol', 'solana'],
    'BNB': ['bnb', 'binance coin', 'binance'],
    'XRP': ['xrp', 'ripple'],
    'ADA': ['ada', 'cardano'],
    'DOGE': ['doge', 'dogecoin'],
    'AVAX': ['avax', 'avalanche'],
    'MATIC': ['matic', 'polygon'],
    'DOT': ['dot', 'polkadot'],
    'LINK': ['link', 'chainlink'],
    'LTC': ['ltc', 'litecoin'],
}

YF_TICKERS = {
    'BTC': 'BTC-USD',
    'ETH': 'ETH-USD',
    'SOL': 'SOL-USD',
    'BNB': 'BNB-USD',
    'XRP': 'XRP-USD',
    'ADA': 'ADA-USD',
    'DOGE': 'DOGE-USD',
    'AVAX': 'AVAX-USD',
    'MATIC': 'MATIC-USD',
    'DOT': 'DOT-USD',
    'LINK': 'LINK-USD',
    'LTC': 'LTC-USD',
}

print('Output:', OUTPUT_DIR.resolve())

## 2. Скачать Kaggle dataset через `kagglehub`

Это ровно тот способ, который нужен для Colab / Jupyter.

In [ ]:
path = kagglehub.dataset_download(KAGGLE_DATASET)
print('Path to dataset files:', path)
DATA_DIR = Path(path)
assert DATA_DIR.exists(), DATA_DIR

## 3. Посмотреть структуру файлов

In [ ]:
def file_size_mb(p: Path) -> float:
    return p.stat().st_size / 1024 / 1024

all_files = [p for p in DATA_DIR.rglob('*') if p.is_file()]
files_df = pd.DataFrame([
    {
        'path': str(p),
        'name': p.name,
        'suffix': p.suffix.lower(),
        'size_mb': round(file_size_mb(p), 3),
    }
    for p in all_files
]).sort_values('size_mb', ascending=False)

display(files_df)
files_df.to_csv(OUTPUT_DIR / 'kaggle_files.csv', index=False)

## 4. Прочитать все подходящие таблицы

Поддерживаются `.csv`, `.json`, `.jsonl`, `.parquet`, `.xlsx`. Если какой-то файл не читается, он попадёт в список ошибок.

In [ ]:
def read_table(path: Path, max_rows=None):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        for enc in ['utf-8', 'utf-8-sig', 'latin1']:
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False, nrows=max_rows)
            except Exception:
                pass
        return pd.read_csv(path, low_memory=False, nrows=max_rows)
    if suffix in ['.json', '.jsonl']:
        try:
            return pd.read_json(path, lines=True)
        except Exception:
            return pd.read_json(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix in ['.xlsx', '.xls']:
        return pd.read_excel(path, nrows=max_rows)
    raise ValueError(f'Unsupported suffix: {suffix}')

supported_suffixes = {'.csv', '.json', '.jsonl', '.parquet', '.xlsx', '.xls'}
tables = {}
read_errors = []

for p in all_files:
    if p.suffix.lower() not in supported_suffixes:
        continue
    try:
        df = read_table(p)
        tables[p.name] = df
        print(f'OK: {p.name}: {df.shape}')
    except Exception as e:
        read_errors.append({'file': str(p), 'error': repr(e)})
        print(f'ERROR: {p.name}: {e}')

if read_errors:
    display(pd.DataFrame(read_errors))

print('Loaded tables:', list(tables.keys()))

## 5. Авто-профиль таблиц

Здесь ноутбук ищет date/time columns, text columns, asset/symbol columns и оценивает, насколько таблица подходит для нашего pipeline.

In [ ]:
DATE_HINTS = ['date', 'time', 'timestamp', 'published', 'created', 'updated']
TEXT_HINTS = ['title', 'text', 'content', 'body', 'summary', 'description', 'headline', 'article']
ASSET_HINTS = ['symbol', 'ticker', 'coin', 'currency', 'asset', 'cryptocurrency', 'category']
SOURCE_HINTS = ['source', 'publisher', 'site', 'domain', 'author']

def lower_cols(df):
    return {str(c).lower(): c for c in df.columns}

def candidate_cols(df, hints):
    out = []
    for c in df.columns:
        name = str(c).lower()
        if any(h in name for h in hints):
            out.append(c)
    return out

def date_parse_rate(s, sample_size=5000):
    sample = s.dropna().astype(str).head(sample_size)
    if len(sample) == 0:
        return 0.0, None, None
    parsed = pd.to_datetime(sample, errors='coerce', utc=True)
    rate = parsed.notna().mean()
    if parsed.notna().any():
        return float(rate), parsed.min(), parsed.max()
    return float(rate), None, None

def text_score(s):
    sample = s.dropna().astype(str).head(5000)
    if len(sample) == 0:
        return {'non_null': 0, 'avg_len': 0, 'median_len': 0}
    lengths = sample.str.len()
    return {
        'non_null': int(s.notna().sum()),
        'avg_len': float(lengths.mean()),
        'median_len': float(lengths.median()),
    }

def count_asset_mentions(text_series):
    sample_text = text_series.fillna('').astype(str).str.lower()
    counts = {}
    for asset, words in ASSET_KEYWORDS.items():
        pattern = r'\b(' + '|'.join(re.escape(w.lower()) for w in words) + r')\b'
        counts[asset] = int(sample_text.str.contains(pattern, regex=True).sum())
    return counts

profiles = []
for name, df in tables.items():
    date_cols = candidate_cols(df, DATE_HINTS)
    text_cols = candidate_cols(df, TEXT_HINTS)
    asset_cols = candidate_cols(df, ASSET_HINTS)
    source_cols = candidate_cols(df, SOURCE_HINTS)

    date_info = []
    for c in date_cols:
        rate, dmin, dmax = date_parse_rate(df[c])
        date_info.append((c, rate, dmin, dmax))
    best_date = max(date_info, key=lambda x: x[1], default=(None, 0, None, None))

    text_info = []
    for c in text_cols:
        ts = text_score(df[c])
        text_info.append((c, ts['non_null'], ts['avg_len'], ts['median_len']))
    best_text = max(text_info, key=lambda x: (x[1], x[2]), default=(None, 0, 0, 0))

    combined_text = pd.Series('', index=df.index, dtype='object')
    for c in text_cols:
        combined_text = combined_text + ' ' + df[c].fillna('').astype(str)
    mention_counts = count_asset_mentions(combined_text) if text_cols else {}

    rows = len(df)
    has_date = best_date[1] > 0.8
    has_text = best_text[0] is not None and best_text[2] >= 15
    total_mentions = sum(mention_counts.values())
    score = 0
    score += min(rows / 10000, 1) * 30
    score += 25 if has_date else 0
    score += 25 if has_text else 0
    score += min(total_mentions / max(rows, 1), 1) * 20

    profiles.append({
        'table': name,
        'rows': rows,
        'cols': df.shape[1],
        'best_date_col': best_date[0],
        'date_parse_rate': round(best_date[1], 3),
        'date_min': best_date[2],
        'date_max': best_date[3],
        'best_text_col': best_text[0],
        'best_text_avg_len': round(best_text[2], 1),
        'text_cols': ', '.join(map(str, text_cols)),
        'asset_cols': ', '.join(map(str, asset_cols)),
        'source_cols': ', '.join(map(str, source_cols)),
        'asset_mentions_total': total_mentions,
        'BTC_mentions': mention_counts.get('BTC', 0),
        'ETH_mentions': mention_counts.get('ETH', 0),
        'SOL_mentions': mention_counts.get('SOL', 0),
        'suitability_score': round(score, 1),
    })

profile_df = pd.DataFrame(profiles).sort_values('suitability_score', ascending=False)
display(profile_df)
profile_df.to_csv(OUTPUT_DIR / 'table_profiles.csv', index=False)

## 6. Выбрать лучшую таблицу и нормализовать events

Если авто-выбор ошибся, вручную поменяй `SELECTED_TABLE`, `DATE_COL`, `TEXT_COLS`, `ASSET_COL`.

In [ ]:
SELECTED_TABLE = profile_df.iloc[0]['table']
selected_profile = profile_df[profile_df['table'] == SELECTED_TABLE].iloc[0]

df0 = tables[SELECTED_TABLE].copy()
DATE_COL = selected_profile['best_date_col']
TEXT_COLS = [c for c in candidate_cols(df0, TEXT_HINTS) if c in df0.columns]
ASSET_COL = None
asset_candidates = candidate_cols(df0, ASSET_HINTS)
if asset_candidates:
    ASSET_COL = asset_candidates[0]

print('SELECTED_TABLE:', SELECTED_TABLE)
print('DATE_COL:', DATE_COL)
print('TEXT_COLS:', TEXT_COLS)
print('ASSET_COL:', ASSET_COL)
display(df0.head())

In [ ]:
def normalize_events(df, date_col, text_cols, asset_col=None):
    out = pd.DataFrame(index=df.index)
    out['event_time'] = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    out['event_date'] = out['event_time'].dt.tz_convert(None).dt.floor('D')

    text = pd.Series('', index=df.index, dtype='object')
    for c in text_cols:
        text = text + ' ' + df[c].fillna('').astype(str)
    out['text'] = text.str.strip()
    out['text_len'] = out['text'].str.len()

    if asset_col is not None and asset_col in df.columns:
        out['asset_raw'] = df[asset_col].fillna('').astype(str)
    else:
        out['asset_raw'] = ''

    # Keyword-based asset labels. One news item can match several assets.
    lower_text = (out['text'] + ' ' + out['asset_raw']).fillna('').str.lower()
    matched = []
    for txt in lower_text:
        labels = []
        for asset, words in ASSET_KEYWORDS.items():
            pattern = r'\b(' + '|'.join(re.escape(w.lower()) for w in words) + r')\b'
            if re.search(pattern, txt):
                labels.append(asset)
        matched.append(labels)
    out['asset_list'] = matched
    out = out.dropna(subset=['event_time'])
    out = out[out['text_len'] > 10].copy()
    return out

events = normalize_events(df0, DATE_COL, TEXT_COLS, ASSET_COL)
print('events:', events.shape)
display(events.head())
events[['event_time', 'event_date', 'text', 'text_len', 'asset_raw', 'asset_list']].to_csv(OUTPUT_DIR / 'normalized_events_sample.csv', index=False)

## 7. Оценить покрытие по активам

Смотрим, по каким монетам больше всего текстов и дат. Это главный фильтр перед ML pipeline.

In [ ]:
asset_rows = []
for asset in ASSET_KEYWORDS:
    mask = events['asset_list'].apply(lambda xs: asset in xs)
    sub = events[mask].copy()
    if sub.empty:
        continue
    daily_counts = sub.groupby('event_date').size()
    asset_rows.append({
        'asset': asset,
        'n_events': int(len(sub)),
        'n_dates': int(sub['event_date'].nunique()),
        'date_min': sub['event_date'].min(),
        'date_max': sub['event_date'].max(),
        'avg_events_per_active_date': round(float(daily_counts.mean()), 2),
        'median_events_per_active_date': round(float(daily_counts.median()), 2),
    })

asset_profile = pd.DataFrame(asset_rows).sort_values(['n_dates', 'n_events'], ascending=False)
display(asset_profile)
asset_profile.to_csv(OUTPUT_DIR / 'asset_text_coverage.csv', index=False)

if not asset_profile.empty:
    ax = asset_profile.head(12).plot(x='asset', y='n_events', kind='bar', legend=False, figsize=(10, 4))
    ax.set_title('News count by asset')
    ax.set_xlabel('asset')
    ax.set_ylabel('n_events')
    plt.tight_layout()
    plt.show()

## 8. Подтянуть prices и проверить пересечение

Пока не запускаем FinBERT. Только проверяем, есть ли достаточно дат для построения таргета.

Для crypto daily target:

```python
y_t = 1{Close_{t+h} > Close_t}
```

In [ ]:
def download_price(asset, start, end):
    yf_ticker = YF_TICKERS.get(asset)
    if yf_ticker is None:
        return pd.DataFrame()
    px = yf.download(yf_ticker, start=start, end=end, auto_adjust=False, progress=False)
    if px.empty:
        return pd.DataFrame()
    if isinstance(px.columns, pd.MultiIndex):
        px.columns = [c[0] for c in px.columns]
    px = px.reset_index()
    px.columns = [str(c).lower().replace(' ', '_') for c in px.columns]
    px = px.rename(columns={'date': 'date', 'adj_close': 'adj_close'})
    px['date'] = pd.to_datetime(px['date']).dt.floor('D')
    return px.sort_values('date').reset_index(drop=True)

def make_daily_text_features(events, asset):
    sub = events[events['asset_list'].apply(lambda xs: asset in xs)].copy()
    if sub.empty:
        return pd.DataFrame()
    daily = sub.groupby('event_date').agg(
        text_count=('text', 'size'),
        avg_text_len=('text_len', 'mean'),
        max_text_len=('text_len', 'max'),
    ).reset_index().rename(columns={'event_date': 'date'})
    daily['has_text'] = 1
    return daily

def evaluate_asset_candidate(asset, events, horizons=(1, 3, 7)):
    daily_text = make_daily_text_features(events, asset)
    if daily_text.empty:
        return []
    start = (daily_text['date'].min() - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
    end = (daily_text['date'].max() + pd.Timedelta(days=max(horizons) + 30)).strftime('%Y-%m-%d')
    px = download_price(asset, start, end)
    if px.empty:
        return []
    rows = []
    merged = px.merge(daily_text, on='date', how='left')
    merged[['text_count', 'avg_text_len', 'max_text_len', 'has_text']] = merged[['text_count', 'avg_text_len', 'max_text_len', 'has_text']].fillna(0)
    for h in horizons:
        temp = merged.copy()
        temp[f'future_close_{h}d'] = temp['close'].shift(-h)
        temp[f'y_{h}d'] = (temp[f'future_close_{h}d'] > temp['close']).astype(float)
        temp = temp.dropna(subset=[f'future_close_{h}d'])
        active = temp[temp['text_count'] > 0]
        if len(active) == 0:
            pos_share = np.nan
        else:
            pos_share = active[f'y_{h}d'].mean()
        rows.append({
            'asset': asset,
            'yf_ticker': YF_TICKERS.get(asset),
            'horizon_days': h,
            'price_rows': int(len(px)),
            'news_dates': int(daily_text['date'].nunique()),
            'overlap_news_price_dates': int(temp[temp['text_count'] > 0]['date'].nunique()),
            'total_news_on_price_dates': int(temp['text_count'].sum()),
            'date_min': temp['date'].min(),
            'date_max': temp['date'].max(),
            'target_positive_share_on_news_dates': round(float(pos_share), 3) if pd.notna(pos_share) else np.nan,
        })
    return rows

candidate_rows = []
for asset in asset_profile['asset'].tolist() if not asset_profile.empty else []:
    candidate_rows.extend(evaluate_asset_candidate(asset, events))

candidate_df = pd.DataFrame(candidate_rows)
if not candidate_df.empty:
    candidate_df['suitability_score'] = (
        np.minimum(candidate_df['overlap_news_price_dates'] / 365, 1) * 40
        + np.minimum(candidate_df['total_news_on_price_dates'] / 5000, 1) * 35
        + (1 - (candidate_df['target_positive_share_on_news_dates'] - 0.5).abs().fillna(0.5) * 2) * 25
    ).round(1)
    candidate_df = candidate_df.sort_values('suitability_score', ascending=False)

display(candidate_df)
candidate_df.to_csv(OUTPUT_DIR / 'price_overlap_candidates.csv', index=False)

if not candidate_df.empty:
    ax = candidate_df.head(15).plot(x='asset', y='suitability_score', kind='bar', legend=False, figsize=(10, 4))
    ax.set_title('Dataset suitability score by asset / horizon')
    ax.set_xlabel('asset')
    ax.set_ylabel('score')
    plt.tight_layout()
    plt.show()
else:
    print('No candidate rows. Check asset detection or date/text columns.')

## 9. Сформировать short-list для будущего pipeline

Выбираем asset/horizon с максимальным числом пересечений news + prices и не слишком перекошенным target.

In [ ]:
if candidate_df.empty:
    shortlist = pd.DataFrame()
else:
    shortlist = candidate_df[
        (candidate_df['overlap_news_price_dates'] >= 30)
        & (candidate_df['total_news_on_price_dates'] >= 100)
    ].copy()
    if shortlist.empty:
        shortlist = candidate_df.head(10).copy()
    else:
        shortlist = shortlist.head(10)

display(shortlist)
shortlist.to_csv(OUTPUT_DIR / 'shortlist_for_run05.csv', index=False)

## 10. Собрать один sample dataset для выбранного кандидата

Это ещё не полный RUN 05. Это проверочный CSV, который показывает, что news + prices можно совместить.

In [ ]:
if not shortlist.empty:
    BEST_ASSET = shortlist.iloc[0]['asset']
    BEST_H = int(shortlist.iloc[0]['horizon_days'])
else:
    BEST_ASSET = 'BTC'
    BEST_H = 1

print('BEST_ASSET:', BEST_ASSET)
print('BEST_HORIZON_DAYS:', BEST_H)

daily_text = make_daily_text_features(events, BEST_ASSET)
start = (daily_text['date'].min() - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
end = (daily_text['date'].max() + pd.Timedelta(days=BEST_H + 30)).strftime('%Y-%m-%d')
px = download_price(BEST_ASSET, start, end)
sample = px.merge(daily_text, on='date', how='left')
sample[['text_count', 'avg_text_len', 'max_text_len', 'has_text']] = sample[['text_count', 'avg_text_len', 'max_text_len', 'has_text']].fillna(0)
sample[f'future_close_{BEST_H}d'] = sample['close'].shift(-BEST_H)
sample['y'] = (sample[f'future_close_{BEST_H}d'] > sample['close']).astype(float)
sample = sample.dropna(subset=[f'future_close_{BEST_H}d']).reset_index(drop=True)

sample_path = OUTPUT_DIR / f'sample_{BEST_ASSET}_{BEST_H}d_dataset.csv'
sample.to_csv(sample_path, index=False)

print('sample shape:', sample.shape)
print('saved:', sample_path)
display(sample.head())
display(sample[['date', 'close', 'volume', 'text_count', 'has_text', 'y']].tail())

## 11. Сохранить исследовательский отчёт

После запуска пришли мне эти файлы:

- `artifacts/run_05_dataset_research/table_profiles.csv`
- `artifacts/run_05_dataset_research/asset_text_coverage.csv`
- `artifacts/run_05_dataset_research/price_overlap_candidates.csv`
- `artifacts/run_05_dataset_research/shortlist_for_run05.csv`

По ним выберем, на чём запускать полный RUN 05.

In [ ]:
notes = []
notes.append('# RUN 05 dataset research notes\n')
notes.append(f'Kaggle dataset: `{KAGGLE_DATASET}`\n')
notes.append(f'Selected table: `{SELECTED_TABLE}`\n')
notes.append(f'Date column: `{DATE_COL}`\n')
notes.append(f'Text columns: `{TEXT_COLS}`\n')
notes.append(f'Asset column: `{ASSET_COL}`\n')
notes.append(f'Normalized events: {len(events)}\n')
notes.append(f'Best asset: `{BEST_ASSET}`\n')
notes.append(f'Best horizon days: `{BEST_H}`\n')

if not shortlist.empty:
    notes.append('\n## Shortlist\n')
    notes.append(shortlist.to_markdown(index=False))

notes_path = OUTPUT_DIR / 'dataset_research_notes.md'
notes_path.write_text('\n'.join(notes), encoding='utf-8')

print('Saved artifacts:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print('-', p)